# 第一階段：粗估圓的半徑 r

**輸入**：一張圖片的路徑
**輸出**：一個數字 r（這張圖裡「主要的圓」半徑大約幾個像素，原圖尺度）

## 怎麼用
1. 把下面「設定」那格的 `IMAGE_PATH` 改成你的圖片
2. 從上到下依序執行每一格（或選單 Run All）

## 做了什麼
把圖縮成 600、300、150 三種大小（影像金字塔），每一種都跑一次：

1. 算邊緣（gx, gy, mag）
2. Hough 用寬鬆條件提名候選圓（寧可多，不要漏）
3. 沿圓周打分數，分辨真圓和假圓
4. NMS：同一個位置只留一個，包在裡面的小圓淘汰

三層找到的圓換回原圖尺度、合併，再 NMS 一次去掉重複的，然後：

5. 面積加權的半徑分布，找最高點 → r

中間資料都存在 `.tmp/<圖片檔名>/` 底下。
結果 r 也會存在 `.tmp/<圖片檔名>/summary.json`，第二個 notebook 會自動讀取。

In [3]:
import os, json, csv
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 如果上面出現 ModuleNotFoundError，先執行：
#   pip install opencv-python numpy matplotlib

## 設定（只要改這裡）

In [4]:
IMAGE_PATH = "sample_02.jpg"   # ← 改成你的圖片路徑
TMP_DIR = ".tmp"                # 中間資料存放的資料夾

## 參數
這些數字是設計時定一次的，不需要每張圖都改。
- 如果你的圓都很大，可以在 `WORK_SIZES` 加入更小的尺寸（例如 75）
- 如果你的圓都很小，可以把最大的尺寸設大一點（例如 1000）

In [5]:
PARAMS = {
    # 影像金字塔：把圖縮成這幾種大小（最長邊像素），每種都找一次圓
    # 為什麼要好幾種？圓太大時，外框在大圖上是一顆一顆的點，打分數會失敗；
    # 縮小後外框才變成一條清楚的線。反過來，小圓要在大圖上才看得清楚。
    # 所以不管圓多大，總有一層剛好適合它。
    "WORK_SIZES": (600, 300, 150),

    # 模糊程度：還不知道 r，只能用固定的輕微模糊，抹掉最細的雜點
    "BLUR": 1.5,

    # Hough 最小半徑（縮小後的像素）。最大半徑自動設成圖的一半
    "HOUGH_R_MIN": 6,
    # Hough 兩個圓心至少隔幾個像素（很小 → 候選很多）
    "HOUGH_MIN_DIST": 4,
    # Hough 至少幾票才算候選（偏低 → 候選很多）
    "HOUGH_VOTES": 25,
    # Canny 邊緣門檻（Hough 內部用）
    "HOUGH_CANNY": 80,

    # 打分數：mag 要比整張圖多少 % 的像素強，才算「強邊緣」
    "EDGE_PERCENTILE": 80,
    # 打分數：邊緣箭頭跟半徑方向夾角的 cos 要大於多少（0.8 ≈ 37 度以內）
    "DIRECTION_COS": 0.8,
    # 打分數：至少幾分才可信。這裡偏嚴格，讓拿去投票的圓比較乾淨
    "MIN_SCORE": 0.7,

    # NMS：圓心距離 < OVERLAP ×（兩半徑相加）→ 重疊太多
    "OVERLAP": 0.75,
    # NMS：圓心距離 < INSIDE × 較大的半徑 → 包在裡面
    "INSIDE": 0.85,

    # 估 r：每座小山的寬度（log 尺度），0.15 ≈ ±15%
    "HILL_WIDTH": 0.15,
    # 至少要幾個圓才估 r，太少就放棄
    "MIN_CIRCLES": 3,
}

## 步驟 1：算邊緣
每個像素算出三個數字：
- `gx`：左右方向亮度變化
- `gy`：上下方向亮度變化
- `mag`：邊緣強度 √(gx² + gy²)

`(gx, gy)` 合起來是一個箭頭，指向變亮的方向。真圓外框上的箭頭都會對著圓心。

In [6]:
def compute_edges(gray, blur):
    """
    算出每個像素的邊緣資訊（三張跟 gray 一樣大的表格）：
        gx   左右方向亮度變化（往右變亮 = 正，變暗 = 負）
        gy   上下方向亮度變化（往下變亮 = 正）
        mag  邊緣強度 √(gx² + gy²)
    (gx, gy) 合起來是一個箭頭，指向「變亮的方向」。
    """
    b = cv2.GaussianBlur(gray, (0, 0), blur)
    gx = cv2.Sobel(b, cv2.CV_32F, 1, 0)       # 一定要 CV_32F，才放得下負數
    gy = cv2.Sobel(b, cv2.CV_32F, 0, 1)
    return gx, gy, cv2.magnitude(gx, gy)

## 步驟 2：Hough 提名候選圓
每個邊緣點沿著箭頭畫線，很多線交會的地方可能是圓心。
這一步條件故意寬鬆，假圓很多沒關係，後面會淘汰。

In [7]:
def hough_candidates(gray, p):
    """
    讓 Hough 盡量多提名。每個邊緣點沿著箭頭畫線，
    很多線交會的地方就是可能的圓心。
    回傳 list of (x, y, r)；一個都沒有就回傳空 list。
    """
    b = cv2.GaussianBlur(gray, (0, 0), p["BLUR"])
    c = cv2.HoughCircles(
        b, cv2.HOUGH_GRADIENT, dp=1,
        minDist=p["HOUGH_MIN_DIST"], param1=p["HOUGH_CANNY"], param2=p["HOUGH_VOTES"],
        minRadius=p["HOUGH_R_MIN"], maxRadius=min(gray.shape) // 2,
    )
    return [] if c is None else c[0].tolist()

## 步驟 3：打分數（以及打分數前的微調）
**打分數**：沿圓周取 72 個檢查點，每個點問兩個問題：
1. 這裡有強邊緣嗎？
2. 邊緣箭頭有沿著半徑方向嗎？

兩個都是「是」才得分。分數 = 得分點 ÷ 檢查點（0～1）。

**微調（refine_center）**：Hough 給的圓心常常偏一點。先從圓心射出 90 條線，
每條線只在 0.85r～1.15r 的環帶裡找最強的邊緣點，再用這些點擬合一個圓。
這樣真圓才不會因為位置偏了而被冤枉扣分。

In [8]:
def score_circle(edges, thr, x, y, r, p, n_points=72, band_px=2):
    """
    沿圓周取 72 個檢查點，每個點問兩個問題：
        一、這裡有強邊緣嗎？（mag > thr）
        二、邊緣箭頭有沿著半徑方向嗎？（指向或背向圓心，不能斜）
    兩個都是「是」才算一分。分數 = 得分點 ÷ 檢查點（0～1）。

    band_px：每個點往內、往外各多看幾格，容許半徑有一點點誤差。
    落在圖外的點不算（不加分也不扣分），被邊界切到的圓才不會吃虧。
    """
    gx, gy, mag = edges
    h, w = mag.shape
    ang = np.linspace(0, 2 * np.pi, n_points, endpoint=False)
    ca, sa = np.cos(ang)[:, None], np.sin(ang)[:, None]
    rs = r + np.arange(-band_px, band_px + 1)[None, :]
    X = np.round(x + ca * rs).astype(int)
    Y = np.round(y + sa * rs).astype(int)

    inside = ((X >= 0) & (X < w) & (Y >= 0) & (Y < h)).all(axis=1)
    if inside.sum() < n_points * 0.5:
        return 0.0

    Xc, Yc = np.clip(X, 0, w - 1), np.clip(Y, 0, h - 1)
    m = mag[Yc, Xc]
    radial = np.abs(gx[Yc, Xc] * ca + gy[Yc, Xc] * sa) / (m + 1e-6)
    hit = ((m > thr) & (radial > p["DIRECTION_COS"])).any(axis=1)
    return float((hit & inside).sum() / inside.sum())

In [9]:
def refine_center(mag, x, y, r, n_rays=90):
    """
    Hough 給的圓心常常偏一點，打分數前先微調，免得真圓被冤枉扣分。
    從圓心射出 90 條線，每條只在 0.85r～1.15r 的環帶裡找最強的邊緣點，
    再用這些點擬合一個最貼合的圓（最小平方法）。修太多就不採用。
    """
    h, w = mag.shape
    rs = np.linspace(0.85 * r, 1.15 * r, 31)
    pts, wts = [], []
    for a in np.linspace(0, 2 * np.pi, n_rays, endpoint=False):
        X = np.round(x + rs * np.cos(a)).astype(int)
        Y = np.round(y + rs * np.sin(a)).astype(int)
        if not ((X >= 0) & (X < w) & (Y >= 0) & (Y < h)).all():
            continue
        v = mag[Y, X]
        j = int(np.argmax(v))
        pts.append((X[j], Y[j]))
        wts.append(v[j])
    if len(pts) < n_rays * 0.5:
        return x, y, r
    P = np.array(pts, float)[np.array(wts) > np.percentile(wts, 25)]
    A = np.c_[2 * P[:, 0], 2 * P[:, 1], np.ones(len(P))]
    cx, cy, c = np.linalg.lstsq(A, (P ** 2).sum(1), rcond=None)[0]
    cr = np.sqrt(c + cx ** 2 + cy ** 2)
    if np.hypot(cx - x, cy - y) > 0.2 * r or abs(cr - r) > 0.2 * r:
        return x, y, r
    return cx, cy, cr

## 步驟 4：NMS
依「分數 × √半徑」排隊，一個一個選。跟已選的圓衝突就淘汰：
- 重疊太多：圓心距離 < OVERLAP × (r1 + r2)
- 包在裡面：圓心距離 < INSIDE × 較大的半徑

In [10]:
def nms(scored, p):
    """
    依「分數 × √半徑」由高到低排隊，一個一個選。
    跟已選的圓衝突就淘汰：
        - 重疊太多：距離 < OVERLAP × (r1 + r2)
        - 包在裡面：距離 < INSIDE × max(r1, r2)   ← 內部小圓就是這樣被淘汰的
    輸入輸出都是 list of dict（含 x, y, r, score）。
    """
    order = sorted(scored, key=lambda c: -(c["score"] * np.sqrt(c["r"])))
    kept = []
    for c in order:
        ok = True
        for k in kept:
            d = np.hypot(c["x"] - k["x"], c["y"] - k["y"])
            if d < p["OVERLAP"] * (c["r"] + k["r"]) or d < p["INSIDE"] * max(c["r"], k["r"]):
                ok = False
                break
        if ok:
            kept.append(c)
    return kept

## 步驟 5：從半徑分布找 r
只用半徑，不用圓心。每個圓在自己的半徑位置放一座小山，**山的高度 = 面積 r²**，
全部疊起來，**最高點**就是 r。

回答的問題是：「哪種大小的圓，佔了這張圖最多的面積？」

In [11]:
def radius_from_distribution(radii, p):
    """
    只用半徑（不用圓心）。
    每個圓在自己的 log 半徑上放一座小山，山的高度 = 面積 r²，
    全部疊起來，最高點（眾數 mode）就是 r。

    回答的問題：「哪種大小的圓佔了這張圖最多的面積？」
    內部小圓、紋路再多，面積也很小，搶不走答案。

    回傳 r，以及畫圖用的 (grid, 不加權曲線, 加權曲線)
    """
    lr = np.log(radii)
    grid = np.linspace(lr.min() - 0.3, lr.max() + 0.3, 600)
    hills = np.exp(-0.5 * ((grid[:, None] - lr[None, :]) / p["HILL_WIDTH"]) ** 2)
    by_count = hills.sum(axis=1)                 # 每個圓 1 票
    by_area = (hills * radii ** 2).sum(axis=1)   # 每個圓的票數 = 面積
    r = float(np.exp(grid[np.argmax(by_area)]))
    return r, (np.exp(grid), by_count, by_area)

## 畫圖小工具（只是存檔給人看，不影響計算）

In [12]:
def _draw(base_bgr, circles, path, color_fn, thickness=1, note=None):
    img = base_bgr.copy()
    for c in circles:
        cv2.circle(img, (int(round(c["x"])), int(round(c["y"]))), max(1, int(round(c["r"]))),
                   color_fn(c), thickness)
    if note:
        cv2.rectangle(img, (0, 0), (img.shape[1], 26), (0, 0, 0), -1)
        cv2.putText(img, note, (6, 19), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)
    cv2.imwrite(path, img)


def _plot_density(curves, radii, r, stats, path):
    x, by_count, by_area = curves
    fig, ax = plt.subplots(3, 1, figsize=(8, 6.5), sharex=True,
                           gridspec_kw={"height_ratios": [1, 2, 2]})
    ax[0].scatter(radii, np.zeros_like(radii), s=30, c="#444", alpha=0.6)
    ax[0].set_title(f"radii of the {len(radii)} circles after NMS "
                    f"(mean {stats['mean']:.0f}, median {stats['median']:.0f})",
                    loc="left", fontsize=10)
    ax[1].plot(x, by_count, color="#1f77b4")
    ax[1].axvline(stats["peak_by_count"], color="#1f77b4", ls="--")
    ax[1].set_title(f"each circle = 1 vote -> peak {stats['peak_by_count']:.1f}",
                    loc="left", fontsize=10)
    ax[2].fill_between(x, by_area, color="#2ca02c", alpha=0.25)
    ax[2].plot(x, by_area, color="#2ca02c")
    ax[2].axvline(r, color="#2ca02c", ls="--")
    ax[2].set_title(f"each circle's vote = area r^2 -> peak r = {r:.1f}  (USED)",
                    loc="left", fontsize=10)
    ax[2].set_xscale("log")
    ticks = [t for t in (5, 10, 20, 30, 50, 70, 100, 150, 200, 300, 500, 700, 1000)
             if x.min() <= t <= x.max()]
    ax[2].set_xticks(ticks)
    ax[2].set_xticklabels([str(t) for t in ticks])
    ax[2].minorticks_off()
    ax[2].set_xlabel("radius (pixels, original image, log scale)")
    for a in ax:
        a.set_yticks([])
        a.spines[["top", "right", "left"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(path, dpi=100)
    plt.close(fig)


def show(path, title=None, width=10):
    """在 notebook 裡顯示一張存好的圖"""
    img = cv2.imread(path)
    if img is None:
        print("找不到圖：", path); return
    h, w = img.shape[:2]
    if title:
        print(title)   # 用 print 顯示標題（matplotlib 預設字型沒有中文）
    plt.figure(figsize=(width, width * h / w))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()

## 金字塔的其中一層
把原圖縮成某個大小，在上面跑步驟 1～4。
**回傳前會把座標除以縮放倍率 s，換回原圖尺度**，這樣三層的結果才能放在一起比較。

In [13]:
def find_circles_one_level(gray_full, s, p, out, tag):
    """
    把原圖縮成 s 倍，在這張小圖上跑步驟 1～4，
    回傳：
        rows  這一層所有候選圓的紀錄（座標已換回原圖）
        kept  這一層 NMS 之後剩下的圓（座標已換回原圖）
    中間圖存成 <tag>_01_gray.png ... <tag>_05_after_nms.jpg
    """
    gray = cv2.resize(gray_full, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    base = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    cv2.imwrite(os.path.join(out, f"{tag}_01_gray.png"), gray)

    # 步驟 1
    edges = compute_edges(gray, p["BLUR"])
    mag = edges[2]
    cv2.imwrite(os.path.join(out, f"{tag}_02_edges.png"),
                np.clip(mag / np.percentile(mag, 99) * 255, 0, 255).astype(np.uint8))
    thr = float(np.percentile(mag, p["EDGE_PERCENTILE"]))

    # 步驟 2
    rows = [{"x": x, "y": y, "r": r, "score": None, "stage": "hough"}
            for x, y, r in hough_candidates(gray, p)]
    _draw(base, rows, os.path.join(out, f"{tag}_03_candidates.jpg"), lambda c: (0, 0, 255),
          note=f"{tag} Hough candidates: {len(rows)}")

    # 步驟 3：先快速粗篩（省時間），再微調圓心，再正式打分數
    for c in rows:
        quick = score_circle(edges, thr, c["x"], c["y"], c["r"], p,
                             n_points=36, band_px=max(2, int(0.06 * c["r"])))
        if quick < p["MIN_SCORE"] - 0.15:
            c["stage"] = "dropped_quick_check"
            continue
        c["x"], c["y"], c["r"] = map(float, refine_center(mag, c["x"], c["y"], c["r"]))
        c["score"] = score_circle(edges, thr, c["x"], c["y"], c["r"], p)
        c["stage"] = "passed_score" if c["score"] >= p["MIN_SCORE"] else "dropped_low_score"
    passed = [c for c in rows if c["stage"] == "passed_score"]
    _draw(base, passed, os.path.join(out, f"{tag}_04_passed_score.jpg"),
          lambda c: (0, int(255 * c["score"]), int(255 * (1 - c["score"]))),
          note=f"{tag} passed score >= {p['MIN_SCORE']}: {len(passed)}")

    # 步驟 4
    kept = nms(passed, p)
    for c in passed:
        c["stage"] = "kept_after_nms" if c in kept else "dropped_nms"
    _draw(base, kept, os.path.join(out, f"{tag}_05_after_nms.jpg"), lambda c: (0, 220, 0), 2,
          note=f"{tag} after NMS: {len(kept)}")

    # 換回原圖座標
    for c in rows:
        c["x"], c["y"], c["r"] = c["x"] / s, c["y"] / s, c["r"] / s
        c["level"] = tag
    return rows, kept

## 主函式：把全部串起來

In [14]:
def estimate_radius(image_path, tmp_dir=".tmp", params=None):
    """
    輸入：圖片路徑
    輸出：r 的估計值（原圖像素，float）。估不出來回傳 None。

    中間資料存在 <tmp_dir>/<圖片檔名>/ 底下（見檔案最上面的說明）。
    params 可以傳入一個 dict，覆蓋部分預設參數，例如 {"MIN_SCORE": 0.6}。
    """
    p = dict(PARAMS, **(params or {}))

    # 準備存檔的資料夾
    name = os.path.splitext(os.path.basename(image_path))[0]
    out = os.path.join(tmp_dir, name)
    os.makedirs(out, exist_ok=True)

    # 讀圖（統一用這種方式轉灰階，每次結果才會一樣）
    gray_full = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if gray_full is None:
        raise FileNotFoundError(image_path)
    H, W = gray_full.shape

    # 金字塔：每一層各跑一次步驟 1～4
    all_rows, level_kept, level_counts = [], [], {}
    for size in p["WORK_SIZES"]:
        s = size / max(H, W)
        if s > 1:
            continue                       # 圖本身比這層還小，不需要放大
        rows, kept = find_circles_one_level(gray_full, s, p, out, tag=f"L{size}")
        all_rows += rows
        level_kept += kept
        level_counts[f"L{size}"] = {
            "hough_candidates": len(rows),
            "passed_score": sum(c["stage"] in ("kept_after_nms", "dropped_nms") for c in rows),
            "after_nms": len(kept),
        }

    # 合併：同一個圓可能在好幾層都被找到，再做一次 NMS，只留分數最高的
    kept = nms(level_kept, p)
    for c in level_kept:
        c["merged"] = c in kept
    base_full = cv2.cvtColor(gray_full, cv2.COLOR_GRAY2BGR)
    _draw(base_full, kept, os.path.join(out, "07_merged.jpg"), lambda c: (0, 220, 0),
          max(2, int(max(H, W) / 300)), note=f"merged from all levels: {len(kept)}")

    # 存每個候選圓的紀錄（座標都是原圖像素）
    with open(os.path.join(out, "candidates.csv"), "w", newline="") as f:
        wr = csv.writer(f)
        wr.writerow(["level", "x", "y", "r", "score", "stage", "used_for_r"])
        for c in all_rows:
            wr.writerow([c["level"], round(c["x"], 2), round(c["y"], 2), round(c["r"], 2),
                         "" if c["score"] is None else round(c["score"], 3), c["stage"],
                         bool(c.get("merged", False))])

    summary = {
        "image": image_path,
        "image_size": [W, H],
        "params": p,
        "counts_per_level": level_counts,
        "circles_after_merge": len(kept),
        "r": None,
    }

    # 步驟 5
    radii = np.array([c["r"] for c in kept])              # 已經是原圖像素
    if len(radii) < p["MIN_CIRCLES"]:
        summary["error"] = f"only {len(radii)} circles after merge, need {p['MIN_CIRCLES']}"
        with open(os.path.join(out, "summary.json"), "w") as f:
            json.dump(summary, f, indent=1)
        return None

    r, curves = radius_from_distribution(radii, p)
    stats = {
        "mean": float(radii.mean()),
        "median": float(np.median(radii)),
        "peak_by_count": float(curves[0][np.argmax(curves[1])]),
    }

    # 可信度：NMS 後的圓裡，有多少比例的「面積」落在 r 的 ±25% 內
    # 接近 1：大部分畫面都是這個大小的圓，r 很可信
    # 偏低：圓的大小很分散（或混了很多雜訊），r 要人看一下
    near = np.abs(np.log(radii / r)) < np.log(1.25)
    support = float((radii[near] ** 2).sum() / (radii ** 2).sum())

    _plot_density(curves, radii, r, stats, os.path.join(out, "08_radius_density.png"))

    summary.update({
        "r": r,
        "support": support,
        "other_estimates": stats,
        "radii_after_nms": sorted(round(float(v), 2) for v in radii),
    })
    with open(os.path.join(out, "summary.json"), "w") as f:
        json.dump(summary, f, indent=1)
    return r

## 執行

In [ ]:
r = estimate_radius(IMAGE_PATH, tmp_dir=TMP_DIR)

name = os.path.splitext(os.path.basename(IMAGE_PATH))[0]
out = os.path.join(TMP_DIR, name)
summary = json.load(open(os.path.join(out, "summary.json")))

if r is None:
    print("估不出 r：", summary.get("error"))
else:
    print(f"r ≈ {r:.1f} 像素（原圖尺度）")
    print(f"可信度 support = {summary['support']:.2f}（越接近 1 越可信）")
    print("每一層 NMS 後剩幾個圓：",
          {k: v["after_nms"] for k, v in summary["counts_per_level"].items()})
    print(f"合併後拿來估 r 的圓：{summary['circles_after_merge']} 個")
    print(f"中間資料：{out}/")

## 看中間結果

In [ ]:
for size in PARAMS["WORK_SIZES"]:
    path = os.path.join(out, f"L{size}_05_after_nms.jpg")
    if os.path.exists(path):
        show(path, f"L{size}：這一層 NMS 之後的圓", width=6)

show(os.path.join(out, "07_merged.jpg"), "三層合併後，拿來估 r 的圓")
show(os.path.join(out, "08_radius_density.png"), "半徑分布（下面那張的最高點就是 r）")